In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

eps = 1e-6

# =========================================================
# v4.6
# v4.0 base + stronger feature engineering with:
# - real seasonality
# - new spectral / hydrological features
# - per-site temporal features using only current/past records
# Keep same RF model as v4.0
# =========================================================

# =========================
# 1. Load training datasets
# =========================
water_quality = pd.read_csv("../../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../../data/terraclimate_features_training.csv")

# =========================
# 2. Convert dates
# =========================
water_quality["Sample Date"] = pd.to_datetime(water_quality["Sample Date"], dayfirst=True)
landsat["Sample Date"] = pd.to_datetime(landsat["Sample Date"], dayfirst=True)
terraclimate["Sample Date"] = pd.to_datetime(terraclimate["Sample Date"], dayfirst=True)

# =========================
# 3. Merge training datasets
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 4. Feature engineering helper
# =========================
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    # -------------------------
    # Basic temporal features
    # -------------------------
    data["month"] = data["Sample Date"].dt.month
    data["year"] = data["Sample Date"].dt.year
    data["dayofyear"] = data["Sample Date"].dt.dayofyear
    data["quarter"] = data["Sample Date"].dt.quarter
    data["weekofyear"] = data["Sample Date"].dt.isocalendar().week.astype(int)

    # Real seasonality
    data["sin_doy"] = np.sin(2 * np.pi * data["dayofyear"] / 365.0)
    data["cos_doy"] = np.cos(2 * np.pi * data["dayofyear"] / 365.0)

    # -------------------------
    # Site identifier / spatial simple features
    # -------------------------
    data["site_id"] = (
        data["Latitude"].round(6).astype(str) + "_" +
        data["Longitude"].round(6).astype(str)
    )

    data["lat_lon_interaction"] = data["Latitude"] * data["Longitude"]
    data["latitude_sq"] = data["Latitude"] ** 2
    data["longitude_sq"] = data["Longitude"] ** 2

    # Sort for temporal-by-site features
    data = data.sort_values(["site_id", "Sample Date"]).reset_index(drop=True)

    # -------------------------
    # Core v4.0 features
    # -------------------------
    data["nir_swir16_ratio"] = data["nir"] / (data["swir16"] + eps)
    data["nir_swir22_ratio"] = data["nir"] / (data["swir22"] + eps)
    data["green_nir_ratio"] = data["green"] / (data["nir"] + eps)

    data["nir_minus_swir16"] = data["nir"] - data["swir16"]
    data["nir_minus_green"] = data["nir"] - data["green"]

    data["ndmi_pet"] = data["NDMI"] * data["pet"]
    data["mndwi_pet"] = data["MNDWI"] * data["pet"]

    data["swir_ratio"] = data["swir16"] / (data["swir22"] + eps)

    data["nir_pet"] = data["nir"] * data["pet"]
    data["swir16_pet"] = data["swir16"] * data["pet"]
    data["ndmi_day"] = data["NDMI"] * data["dayofyear"]

    # -------------------------
    # New spectral features
    # -------------------------
    data["nir_green_ratio"] = data["nir"] / (data["green"] + eps)
    data["green_nir_diffnorm"] = (data["green"] - data["nir"]) / (data["green"] + data["nir"] + eps)

    data["swir16_nir_ratio"] = data["swir16"] / (data["nir"] + eps)
    data["swir22_nir_ratio"] = data["swir22"] / (data["nir"] + eps)

    data["swir_slope"] = data["swir22"] - data["swir16"]
    data["nir_swir_sum"] = data["nir"] + data["swir16"] + data["swir22"]

    data["water_contrast_index"] = data["MNDWI"] - data["NDMI"]
    data["water_moisture_product"] = data["MNDWI"] * data["NDMI"]

    data["green_swir16_ratio"] = data["green"] / (data["swir16"] + eps)
    data["green_swir22_ratio"] = data["green"] / (data["swir22"] + eps)

    data["normalized_swir_difference"] = (data["swir16"] - data["swir22"]) / (data["swir16"] + data["swir22"] + eps)

    data["band_mean_4"] = data[["nir", "green", "swir16", "swir22"]].mean(axis=1)
    data["band_std_4"] = data[["nir", "green", "swir16", "swir22"]].std(axis=1)

    data["swir_total"] = data["swir16"] + data["swir22"]
    data["water_purity_index"] = data["MNDWI"] / (np.abs(data["NDMI"]) + eps)

    # -------------------------
    # Second-order spectral features
    # -------------------------
    data["ndmi_sq"] = data["NDMI"] ** 2
    data["mndwi_sq"] = data["MNDWI"] ** 2
    data["ndmi_x_mndwi"] = data["NDMI"] * data["MNDWI"]
    data["nir_x_swir16"] = data["nir"] * data["swir16"]
    data["green_x_mndwi"] = data["green"] * data["MNDWI"]

    # -------------------------
    # Hydrological proxy features
    # -------------------------
    data["relative_lowflow_index"] = data["pet"] / (data["MNDWI"] + 1.5)
    data["dilution_concentration_index"] = data["MNDWI"] / (data["pet"] + eps)
    data["hydrologic_stress_index"] = data["pet"] * (1 - data["MNDWI"])
    data["sediment_mobilization_proxy"] = data["swir_total"] / (data["MNDWI"] + 1.5)
    data["channel_exposure_index"] = data["swir_total"] - data["MNDWI"]
    data["evapoconcentration_proxy"] = data["pet"] * data["swir22_nir_ratio"]
    data["riparian_influence_proxy"] = data["NDMI"] - data["MNDWI"]

    # -------------------------
    # Climate transforms
    # -------------------------
    pet_mean = data["pet"].mean()
    pet_std = data["pet"].std() + eps

    data["pet_zscore"] = (data["pet"] - pet_mean) / pet_std
    data["pet_log"] = np.log1p(np.clip(data["pet"], a_min=0, a_max=None))
    data["pet_sq"] = data["pet"] ** 2
    data["pet_sin_doy"] = data["pet"] * data["sin_doy"]
    data["pet_cos_doy"] = data["pet"] * data["cos_doy"]

    # -------------------------
    # Per-site temporal features
    # Only based on current/past rows
    # -------------------------
    grp = data.groupby("site_id", group_keys=False)

    data["elapsed_days"] = (
        data["Sample Date"] - grp["Sample Date"].transform("min")
    ).dt.days

    data["days_since_last_sample_at_site"] = grp["Sample Date"].diff().dt.days
    data["days_since_last_sample_at_site"] = data["days_since_last_sample_at_site"].fillna(-1)

    data["site_obs_number"] = grp.cumcount() + 1
    data["site_obs_count"] = grp["site_id"].transform("count")

    # Rolling means using only current/past information
    data["rolling_mndwi_3"] = (
        grp["MNDWI"]
        .rolling(3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    data["rolling_mndwi_5"] = (
        grp["MNDWI"]
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    data["rolling_pet_3"] = (
        grp["pet"]
        .rolling(3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    data["rolling_pet_5"] = (
        grp["pet"]
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Lag features
    data["lag1_mndwi"] = grp["MNDWI"].shift(1)
    data["lag2_mndwi"] = grp["MNDWI"].shift(2)
    data["lag1_pet"] = grp["pet"].shift(1)

    # Site anomalies relative to site mean
    data["mndwi_site_mean"] = grp["MNDWI"].transform("mean")
    data["pet_site_mean"] = grp["pet"].transform("mean")

    data["mndwi_anomaly_site"] = data["MNDWI"] - data["mndwi_site_mean"]
    data["pet_anomaly_site"] = data["pet"] - data["pet_site_mean"]

    # Additional interactions
    data["sin_doy_x_mndwi"] = data["sin_doy"] * data["MNDWI"]
    data["cos_doy_x_ndmi"] = data["cos_doy"] * data["NDMI"]
    data["pet_x_swir_total"] = data["pet"] * data["swir_total"]
    data["pet_x_water_contrast"] = data["pet"] * data["water_contrast_index"]

    # Cleanup
    data.replace([np.inf, -np.inf], np.nan, inplace=True)

    return data

# =========================
# 5. Apply feature engineering to train
# =========================
df = add_features(df)

# =========================
# 6. Handle missing values
# =========================
train_medians = df.median(numeric_only=True)
df.fillna(train_medians, inplace=True)

# =========================
# 7. Define training features and targets
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

drop_cols = targets + ["Sample Date", "Latitude", "Longitude", "site_id"]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[targets]

X_train_medians = X.median(numeric_only=True)

# =========================
# 8. Train final model
# =========================
rf_final = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

# =========================
# 9. Load submission datasets
# =========================
submission = pd.read_csv("../../data/submission_template.csv")
landsat_val = pd.read_csv("../../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../../data/terraclimate_features_validation.csv")

# =========================
# 10. Convert dates
# =========================
submission["Sample Date"] = pd.to_datetime(submission["Sample Date"], dayfirst=True)
landsat_val["Sample Date"] = pd.to_datetime(landsat_val["Sample Date"], dayfirst=True)
terraclimate_val["Sample Date"] = pd.to_datetime(terraclimate_val["Sample Date"], dayfirst=True)

# =========================
# 11. Merge validation datasets
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 12. Apply same feature engineering to validation
# =========================
df_val = add_features(df_val)

# =========================
# 13. Handle missing values
# =========================
df_val.fillna(train_medians, inplace=True)

# =========================
# 14. Prepare validation features
# =========================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "site_id",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

# Align columns exactly with training
X_val = X_val.reindex(columns=X.columns)
X_val = X_val.fillna(X_train_medians)

# =========================
# 15. Predict
# =========================
predictions = rf_final.predict(X_val)

# =========================
# 16. Build submission
# =========================
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

submission_v4_6 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

# =========================
# 17. Export
# =========================
submission_v4_6.to_csv("../../submissions/submission_v4.6.csv", index=False)

# =========================
# 18. Quick check
# =========================
print(submission_v4_6.shape)
print(submission_v4_6.head())
print(submission_v4_6.isna().sum())

(200, 6)
   Longitude   Latitude Sample Date  Total Alkalinity  Electrical Conductance  \
0  27.822778 -32.043333  2014-09-01        144.560152              501.586340   
1  26.077500 -33.329167  2015-09-16        167.303078              554.004560   
2  27.640028 -32.991639  2015-05-07        161.176153              556.196760   
3  24.439167 -34.096389  2012-02-07        170.075504              545.075133   
4  28.581667 -32.000556  2014-10-01        187.980253              578.712380   

   Dissolved Reactive Phosphorus  
0                      50.758000  
1                      54.005000  
2                      47.669000  
3                      57.972667  
4                      54.734333  
Longitude                        0
Latitude                         0
Sample Date                      0
Total Alkalinity                 0
Electrical Conductance           0
Dissolved Reactive Phosphorus    0
dtype: int64
